# Buoy Data Exploration

Load and validate data from Wilmette (45174) and Chicago (45198) buoys.

This notebook demonstrates that the data loader correctly:
- Parses NDBC text format
- Converts wind speeds to knots
- Handles missing values
- Creates proper timestamps

In [ ]:
import sys
sys.path.append('..')

from src.buoy_loader import load_buoy_data
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

## Load Wilmette Buoy Data (45174)

In [ ]:
wilmette_2024 = load_buoy_data('../data/raw/wilmette/45174h2024.txt')
print(f"Loaded {len(wilmette_2024)} rows from Wilmette 2024")
wilmette_2024.head()

## Load Chicago Buoy Data (45198)

In [ ]:
import glob
chicago_files = sorted(glob.glob('../data/raw/chicago/*.txt'))
print(f"Found {len(chicago_files)} Chicago buoy files")
if chicago_files:
    chicago_2024 = load_buoy_data(chicago_files[-1])  # Load most recent
    print(f"Loaded {len(chicago_2024)} rows from Chicago {chicago_files[-1]}")
    chicago_2024.head()
else:
    print("No Chicago files found")

## Data Quality Summary - Wilmette

In [ ]:
print("Date Range:")
print(f"  Start: {wilmette_2024['timestamp'].min()}")
print(f"  End: {wilmette_2024['timestamp'].max()}")
print(f"  Duration: {wilmette_2024['timestamp'].max() - wilmette_2024['timestamp'].min()}")

print("\nData Completeness:")
for col in ['wind_dir_deg', 'wind_speed_knots', 'gust_speed_knots', 'WVHT', 'PRES']:
    completeness = (1 - wilmette_2024[col].isna().sum() / len(wilmette_2024)) * 100
    print(f"  {col}: {completeness:.1f}%")

## Wind Direction Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(wilmette_2024['wind_dir_deg'].dropna(), bins=36, edgecolor='black')
axes[0].set_xlabel('Wind Direction (degrees)')
axes[0].set_ylabel('Count')
axes[0].set_title('Wilmette Wind Direction Distribution')
axes[0].axvspan(315, 360, alpha=0.2, color='blue', label='North Winds')
axes[0].axvspan(0, 45, alpha=0.2, color='blue')
axes[0].legend()

# Polar plot
ax_polar = plt.subplot(122, projection='polar')
theta = wilmette_2024['wind_dir_deg'].dropna() * (3.14159 / 180)
ax_polar.hist(theta, bins=36)
ax_polar.set_theta_zero_location('N')
ax_polar.set_theta_direction(-1)
ax_polar.set_title('Wind Rose')

plt.tight_layout()
plt.show()

## Wind Speed Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(wilmette_2024['wind_speed_knots'].dropna(), bins=50, alpha=0.7, label='Wind Speed', edgecolor='black')
ax.hist(wilmette_2024['gust_speed_knots'].dropna(), bins=50, alpha=0.7, label='Gust Speed', edgecolor='black')
ax.axvline(5, color='red', linestyle='--', label='5 knots threshold')
ax.set_xlabel('Speed (knots)')
ax.set_ylabel('Count')
ax.set_title('Wilmette Wind Speed Distribution (converted from m/s)')
ax.legend()
plt.show()

print("Wind Speed Statistics:")
print(wilmette_2024['wind_speed_knots'].describe())

## Data Validation Checks

In [ ]:
print("Column Types:")
print(wilmette_2024.dtypes)

print("\nWind Direction Range Check:")
valid_dirs = wilmette_2024['wind_dir_deg'].dropna()
in_range = ((valid_dirs >= 0) & (valid_dirs <= 360)).all()
print(f"  All directions in 0-360 range: {in_range}")
if not in_range:
    print(f"  Out of range values: {valid_dirs[(valid_dirs < 0) | (valid_dirs > 360)]}")

print("\nTimestamp Check:")
print(f"  Type: {wilmette_2024['timestamp'].dtype}")
print(f"  Valid timestamps: {wilmette_2024['timestamp'].notna().all()}")

print("\nMissing Value Handling:")
print(f"  VIS has NaN values: {wilmette_2024['VIS'].isna().any()}")
print(f"  TIDE has NaN values: {wilmette_2024['TIDE'].isna().any()}")
print(f"  APD has NaN values: {wilmette_2024['APD'].isna().any()}")

## Example Rows

In [ ]:
# Show a few representative rows with key columns
cols_to_show = ['timestamp', 'wind_dir_deg', 'wind_speed_knots', 'gust_speed_knots', 'WVHT', 'PRES', 'ATMP']
wilmette_2024[cols_to_show].head(10)

## Summary

Data loader successfully:
- Parses NDBC text format into DataFrame
- Creates proper timestamps from date/time columns
- Converts wind speeds from m/s to knots (multiply by 1.94384)
- Replaces missing value sentinels (99.00, 999.0) with NaN
- Validates wind direction is in 0-360 degree range
- Uses domain glossary names (wind_dir_deg, wind_speed_knots)

Data is clean and ready for analysis.